# Multi-model UNSEEN spatial analysis figures

This notebook can be run using papermill, for example:

```bash
papermill -p metric txx -p project txx -p obs_config_file AGCD-CSIRO_r05_tasmax_config.mk -p obs AGCD spatial_analysis_multimodel.ipynb project-txx/spatial_analysis_multimodel_txx.ipynb

papermill -p metric rx1day -p project rx1day -p obs_config_file AGCD-CSIRO_r05_precip_config.mk -p obs AGCD -p bc multiplicative spatial_analysis_multimodel.ipynb project-rx1day/spatial_analysis_multimodel_rx1day.ipynb
```

In [11]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt  # noqa
import numpy as np
from pathlib import Path
import xarray as xr
import sys

sys.path.append("/g/data/xv83/unseen-projects/code")
from unseen import general_utils
from spatial_plots_multimodel import *


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
plt.rcParams["figure.figsize"] = [14, 10]
plt.rcParams["figure.dpi"] = 300
plt.rcParams["figure.constrained_layout.use"] = True
plt.rcParams["contour.linewidth"] = 0.3
plt.rcParams["hatch.color"] = "k"
plt.rcParams["hatch.linewidth"] = 0.5
plt.rcParams["xtick.major.size"] = 4.5
plt.rcParams["xtick.minor.size"] = 3
plt.rcParams["ytick.major.size"] = 4.5
plt.rcParams["ytick.minor.size"] = 3

In [ ]:
# Default parameters
bc = None

In [4]:
metric = "txx"
obs_config_file = "AGCD-CSIRO_r05_tasmax_config.mk"
obs = "AGCD"
bc = "additive"
project = "txx"

In [5]:
# Required parameters
kwargs = locals()
assert "metric" in kwargs, "Must provide a metric name (e.g., txx)"
assert (
    "obs_config_file" in kwargs
), "Must provide a obs_config_file name (e.g., AGCD-tasmax_config.mk)"
assert "obs" in kwargs, "Must provide a observation data set name (e.g., AGCD)"
if "project" not in kwargs:
    kwargs["project"] = metric

In [6]:
if metric == "txx":
    models = np.array(
        [
            "CAFE",
            "BCC-CSM2-MR",
            "CanESM5",
            "CMCC-CM2-SR5",
            "EC-Earth3",
            "IPSL-CM6A-LR",
            "MIROC6",
            "MPI-ESM1-2-HR",
            "MRI-ESM2-0",
            "NorCPM1",
        ]
    )
elif metric == "rx1day":
    models = np.array(
        [
            "BCC-CSM2-MR",
            "CanESM5",
            "CMCC-CM2-SR5",
            "EC-Earth3",
            "HadGEM3-GC31-MM",
            "IPSL-CM6A-LR",
            "MIROC6",
            "MPI-ESM1-2-HR",
            "MRI-ESM2-0",
            "NorCPM1",
        ]
    )

In [7]:
# Get variables from makefile (nested dictionary)
var_dict = get_makefile_vars(
    models, metric, obs, obs_config_file=obs_config_file, project=project
)

In [10]:
# Extract some variables from the dictionaries
plot_dict = eval(var_dict[obs]["plot_dict"])
map_plot_kwargs = plot_dict.pop("map_plot_kwargs")
var = var_dict[obs]["var"]
time_agg = var_dict[obs]["time_agg"]
covariate_base = int(var_dict[obs]["covariate_base"])
covariates = eval(var_dict[obs]["gev_trend_period"])
plot_dict["fig_dir"] = Path(var_dict[obs]["fig_dir"]) / "multimodel"
plot_dict["filestem"] = filestem
plot_dict["filestem_no_bc"] = filestem.split("_bias")[0]
plot_dict["models"] = models

plot_dict_avg = plot_dict.copy()
stability_kwargs = {}
stability_anom_kwargs = {}

if metric == "txx":
    std_dev_ticks = np.arange(0, 4.5, 0.5)  # Median absolute deviation
    if bc is None:
        plot_dict_avg["ticks"] = np.arange(22, 52 + 4, 4)
        plot_dict["ticks"] = np.arange(24, 66 + 4, 4)
    else:
        plot_dict_avg["ticks"] = np.arange(22, 52 + 4, 4)
        plot_dict["ticks"] = np.arange(32, 56 + 2, 2)

    stability_anom_kwargs = dict(
        ticks=np.arange(-3.3, 3.5, 0.2),
        ticklabels=np.around(np.arange(-3.2, 3.4, 0.2), 1),
    )

elif metric == "rx1day":
    std_dev_ticks = np.arange(0, 50 + 10, 10)  # Median absolute deviation
    if bc is None:
        plot_dict_avg["ticks"] = np.arange(0, 200 + 20, 20)
        plot_dict["ticks"] = np.arange(0, 400 + 50, 50)
    else:
        plot_dict_avg["ticks"] = np.arange(0, 120 + 20, 20)
        plot_dict_avg["ticks_anom"] = np.arange(-11, 11 + 2, 2)
        plot_dict["ticks"] = np.arange(0, 450 + 50, 50)
        plot_dict["ticks_anom"] = np.arange(-170, 170 + 40, 40)

In [13]:
# Filestem for figures and datatree
filestem = f"{metric}_{var_dict[obs]['timescale']}_{var_dict[obs]['region']}"
filestem_bc = filestem + f"_bias-corrected-{var_dict[obs]['obs_dataset']}-{bc}"

dt_file = f"{var_dict[obs]['project_dir']}/data/datatree_{filestem}.nc"
dt_file_bc = f"{var_dict[obs]['project_dir']}/data/datatree_{filestem_bc}.nc"

In [14]:
# Create/open datatree of all model and observation datasets
if Path(dt_file).exists():
    # dt_no_bc = xr.open_datatree(dt_file)
    dt = xr.open_datatree(dt_file_bc)
else:
    print("Need to create datatree files first.")
dt

<xarray.DataTree>
Group: /
├── Group: /obs
│   └── Group: /obs/AGCD
│           Dimensions:     (time: 63, lat: 70, lon: 89, dparams: 5)
│           Coordinates:
│             * lat         (lat) float64 560B -44.5 -44.0 -43.5 -43.0 ... -11.0 -10.5 -10.0
│             * lon         (lon) float64 712B 112.0 112.5 113.0 113.5 ... 155.0 155.5 156.0
│             * time        (time) datetime64[ns] 504B 1961-06-30 1962-06-30 ... 2023-06-30
│               event_time  (time, lat, lon) datetime64[ns] 3MB ...
│             * dparams     (dparams) <U10 200B 'c' 'location_0' ... 'scale_0' 'scale_1'
│           Data variables:
│               tasmax      (time, lat, lon) float32 2MB ...
│               dparams_ns  (lat, lon, dparams) float64 249kB ...
│           Attributes: (12/34)
│               url:                           http://www.bom.gov.au/climate/
│               geospatial_lon_min:            111.975
│               bom-cmp-awap_version:          bom-cmp-awap-1.00-100.1.x86_64
│               id:                            Australian Gridded Climate Data (AGCD)
│               keywords:                      Earth Science, Atmosphere, Atmospheric Tem...
│               time_coverage_start:           1910-01-01T09:00:00
│               ...                            ...
│               geospatial_lat_min:            -44.525
│               analysis_components:           mean: the gridded temperature value
│               netcdf_version:                4.3.0 of Jul 16 2013 05:46:56 $
│               date_created:                  2018-08-16T23:46:11.477864
│               licence:                       Copyright for any data supplied by the Bur...
│               cdm_data_type:                 Grid
└── Group: /model
    ├── Group: /model/CAFE
    │       Dimensions:          (sample: 34944, lat: 19, lon: 19, dparams: 5, month: 2,
    │                             lead_time: 9, quantile: 2, bounds: 2)
    │       Coordinates:
    │         * lon              (lon) float64 152B 111.2 113.8 116.2 ... 151.2 153.8 156.2
    │         * lat              (lat) float64 152B -45.51 -43.48 -41.46 ... -11.12 -9.101
    │         * dparams          (dparams) <U10 200B 'c' 'location_0' ... 'scale_1'
    │         * lead_time        (lead_time) int64 72B 0 1 2 3 4 5 6 7 8
    │         * month            (month) int64 16B 5 11
    │         * quantile         (quantile) float64 16B 0.005 0.995
    │         * bounds           (bounds) <U5 40B 'lower' 'upper'
    │           event_time       (sample, lat, lon) object 101MB ...
    │           ensemble         (sample) int64 280kB ...
    │           init_date        (sample) object 280kB ...
    │           time             (sample) object 280kB ...
    │       Dimensions without coordinates: sample
    │       Data variables:
    │           tasmax           (sample, lat, lon) float32 50MB ...
    │           ks_pval          (lat, lon) float32 1kB ...
    │           pval_mask        (lat, lon) bool 361B ...
    │           dparams_ns       (lat, lon, dparams) float64 14kB ...
    │           covariate        (sample) int64 280kB ...
    │           r                (month, lead_time, lat, lon) float64 52kB ...
    │           ci               (month, quantile, lat, lon) float64 12kB ...
    │           min_lead         (month, lat, lon) int64 6kB ...
    │           min_lead_median  (month) float64 16B ...
    │           ci_median        (lat, lon, bounds) float64 6kB ...
    │           ci_aep           (lat, lon, bounds) float64 6kB ...
    │       Attributes:
    │           filename:   atmos_hybrid_daily.zarr
    │           grid_tile:  N/A
    │           grid_type:  regular
    │           title:      AccessOcean-AM2
    │           history:    Thu May 15 04:26:31 2025: /g/data/xv83/as3189/conda/envs/unse...
    ├── Group: /model/BCC-CSM2-MR
    │       Dimensions:          (sample: 3888, lat: 33, lon: 40, dparams: 5, month: 1,
    │                             lead_time: 9, quanti

In [15]:
# Create nested dict of class instance containing variables and datasets
info = {}
info[obs] = InfoSet(
    name=obs,
    file=var_dict[obs]["metric_obs"],
    obs_name=obs,
    ds=dt[f"obs/{obs}"].ds,
    obs_ds=dt[f"obs/{obs}"].ds,
    bias_correction=bc,
    pval_mask=None,
    **plot_dict,
)

for m in models:
    info[m] = InfoSet(
        name=m,
        obs_name=obs,
        file=var_dict[m]["metric_fcst"],
        ds=dt[f"model/{m}"].ds,
        obs_ds=subset_obs_dataset(dt[f"obs/{obs}"].ds, dt[f"model/{m}"].ds),
        pval_mask=dt[f"model/{m}"].ds.pval_mask,
        bias_correction=bc,
        **plot_dict,
    )
    info[m].regridder = shared_grid_regridder(info[m].ds, method="conservative")

for m in info.keys():
    info[m].gev_mask = get_gev_mask(info, m, var_dict, test=var_dict[m]["gev_test"])

/g/data/xv83/as3189/conda/envs/unseen/lib/python3.11/site-packages/xesmf/frontend.py:95: UserWarning: Variables {'lon_bnds'} not found in object but are referred to in the CF attributes.
  lon_bnds = ds.cf.get_bounds('longitude')
/g/data/xv83/as3189/conda/envs/unseen/lib/python3.11/site-packages/xesmf/frontend.py:95: UserWarning: Variables {'lon_bnds'} not found in object but are referred to in the CF attributes.
  lon_bnds = ds.cf.get_bounds('longitude')
/g/data/xv83/as3189/conda/envs/unseen/lib/python3.11/site-packages/xesmf/frontend.py:95: UserWarning: Variables {'lon_bnds'} not found in object but are referred to in the CF attributes.
  lon_bnds = ds.cf.get_bounds('longitude')
/g/data/xv83/as3189/conda/envs/unseen/lib/python3.11/site-packages/xesmf/frontend.py:95: UserWarning: Variables {'lon_bnds'} not found in object but are referred to in the CF attributes.
  lon_bnds = ds.cf.get_bounds('longitude')
/g/data/xv83/as3189/conda/envs/unseen/lib/python3.11/site-packages/xesmf/fronten

## Plots


### Single grid point

In [ ]:
name = "Melbourne"
model = "MPI-ESM1-2-HR"
coords = [-37.8, 145]
da = info[model].ds[var].sel(lat=coords[0], lon=coords[1], method="nearest")
da_obs = info[obs].obs_ds[var].sel(lat=coords[0], lon=coords[1], method="nearest")

In [ ]:

# AGCD & model time series (scatter)
ax = general_utils.plot_timeseries_scatter(
    da,
    da_obs,
    title=f"{name} hottest day of the year ({metric})",
    label=model,
    obs_label='AGCD',
    units=units,
    time_dim="time",
    outfile=f"{home}/figures/{index.lower()}_timeseries_{name.lower()}_{model}.png",
)

In [ ]:
if bc is None:
    # Stability (don't plot for diff bc)
    for method in ["aep", "median"]:
        plot_stability(
            info, var_dict, plot_dict, map_plot_kwargs, method, anomaly=False
        )
        plot_stability(
            info,
            var_dict,
            plot_dict,
            map_plot_kwargs,
            method=method,
            anomaly=True,
            **stability_anom_kwargs,
        )

### Metric maximum and median

In [ ]:
plot_time_agg(info, var, "maximum", plot_dict, map_plot_kwargs)

In [ ]:
plot_time_agg(info, var, "median", plot_dict_avg, map_plot_kwargs)

### Metric maximum (subsampled)

In [ ]:
plot_time_agg_subsampled(info, obs, "maximum", plot_dict, map_plot_kwargs, 10000)

### Model minus observation anomalies

In [ ]:
for anom in ["anom", "anom_pct", "anom_std", "anom_2000yr"]:
    plot_obs_anom(
        info, obs, var, "maximum", anom, covariate_base, plot_dict, map_plot_kwargs
    )

In [ ]:
plot_obs_anom(
    info, obs, var, "median", "anom", covariate_base, plot_dict_avg, map_plot_kwargs
)

### Seasonality/event year

In [ ]:
cmap = month_cmap  # _alt if metric == "txx" else month_cmap

In [ ]:
plot_event_month_mode(
    info,
    plot_dict,
    map_plot_kwargs,
    cmap=cmap,
    add_labels=True if metric == "txx" else False,
)

In [ ]:
plot_record_event_month(info, plot_dict, map_plot_kwargs, time_agg=time_agg, cmap=cmap)

In [ ]:
plot_event_year(
    info, var, time_agg, plot_dict, map_plot_kwargs, ticks=np.arange(1960, 2025, 5)
)

In [ ]:
plot_metric_variability(info, var, plot_dict, map_plot_kwargs, ticks=std_dev_ticks)

### GEV parameters

In [ ]:
# GEV/empirical
for param in ["c", "location_0", "location_1", "scale_0", "scale_1"]:
    plot_nonstationary_gev_param(info, param, plot_dict, map_plot_kwargs)

### Annual exceedance probability (AEP)

In [ ]:
aep = 1

In [ ]:
plot_aep_empirical(info, plot_dict, map_plot_kwargs, var, aep=aep)

In [ ]:
plot_aep(info, plot_dict, map_plot_kwargs, covariates[-1], aep=aep)

In [ ]:
plot_aep_trend(
    info,
    plot_dict,
    map_plot_kwargs,
    covariates,
    aep=aep,
)

# Cumulative annual exceedance probability

In [ ]:
plot_abstract(
    info, plot_dict, map_plot_kwargs, time_agg=time_agg, start_year=2025, n_years=10
)  # For graphic abstract (multi-model median only)

In [ ]:
plot_new_record_probability_empirical(
    info, plot_dict, map_plot_kwargs, var, time_agg=time_agg, n_years=10
)

In [ ]:
plot_new_record_probability(
    info, plot_dict, map_plot_kwargs, covariate_base, time_agg=time_agg, n_years=10
)

In [ ]:
# Multi-model median and median absolute deviation (MAD)
plot_new_record_probability_model_spread(
    info,
    plot_dict,
    map_plot_kwargs,
    covariate_base,
    time_agg=time_agg,
    start_year=2025,
    n_years=10,
    mad_ticks=np.arange(0, 35, 5),
)

In [ ]:
plot_obs_ari(
    info,
    plot_dict,
    map_plot_kwargs,
    var,
    obs,
    covariate_base,
    time_agg=time_agg,
)